# Qwen2.5-VL-3B 4-bit QLoRA Fine-Tuning Guide (Colab & Kaggle Free GPUs)

## 1. Hardware Budget & Model Selection
- **Target Free Environments:** Google Colab (Tesla T4 ~15GB VRAM) & Kaggle (Tesla P100 / T4 ~16GB VRAM).
- **Selected Base Model:** `Qwen/Qwen2.5-VL-3B-Instruct`
- **Why 3B with 4-bit QLoRA?**
  - A 7B vision-language model in 4-bit occupies ~4.5-5GB resting VRAM. However, high-resolution Marathi document images (like 7-12 extracts / सातबारा उतारा) decompose into hundreds of visual tokens. Adding backward pass activations, gradients, and optimizer states triggers CUDA Out-Of-Memory (OOM) on a 15-16GB card.
  - `Qwen2.5-VL-3B-Instruct` in 4-bit occupies only **~2.2 - 2.8 GB** resting VRAM, leaving **12GB+ of headroom** for document image patches, long context sequences, and LoRA gradients.

---
## 2. Environment Setup
Install required dependencies for quantized multimodal loading and fine-tuning.

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes peft qwen-vl-utils trl torchvision

## 3. Loading Qwen2.5-VL-3B in 4-bit Precision
We use `BitsAndBytesConfig` (NormalFloat4 / NF4 with double quantization and bfloat16/float16 compute dtype) and configure pixel bounds to prevent image token explosion.

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

# 1. Define 4-bit quantization configuration (QLoRA)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,        # Nested quantization saves extra memory
    bnb_4bit_quant_type="nf4"              # NormalFloat4
)

# 2. Load processor with resolution constraints for dense documents
min_pixels = 256 * 28 * 28   # Avoids extreme downsampling of small text
max_pixels = 1280 * 28 * 28  # Bounds maximum visual patches to avoid VRAM OOM
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=min_pixels,
    max_pixels=max_pixels
)

# 3. Load 4-bit quantized base model
print(f"Downloading and loading {model_id} in 4-bit NF4...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=compute_dtype,
    quantization_config=quantization_config
)
print("Model loaded successfully in 4-bit!")

## 4. Configuring LoRA (Low-Rank Adaptation)
We freeze the 4-bit quantized base model and attach lightweight trainable LoRA adapters targeting the attention projection matrices.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare quantized model for k-bit training
model = prepare_model_for_kbit_training(model)

# 2. LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Wrap model with LoRA adapters
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

## 5. Next Steps: Dataset Preparation & SFTTrainer
1. Place your annotated Marathi 7-12 extracts in `AI_Model/data/`.
2. Prepare conversations in chat format with user prompt requesting structured JSON extraction and assistant response with ground truth JSON.
3. Use `SFTTrainer` with gradient checkpointing enabled to train comfortably within free-tier GPU limits.